In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/station_list.csv
/kaggle/input/weather.csv
/kaggle/input/usage_frequency.csv
/kaggle/input/daily_rent_detail.csv


In [2]:
!pip install meteostat

ERROR: Could not find a version that satisfies the requirement meteostat (from versions: none)
ERROR: No matching distribution found for meteostat


 # CAPITAL BIKESHARE DEMAND PREDICTION - COMPLETE ANALYSIS
 ##     Advanced Time Series Forecasting with Machine Learning
 
 **Objective**: Predict hourly bikeshare demand using weather, temporal patterns, and historical data
 
 **Key Features**:
 - Robust data cleaning with COVID-era removal
 - Weather data leakage prevention
 - Advanced feature engineering (cyclical encoding, lag features, interactions)
 - Multiple ML models (Linear Regression, Random Forest, XGBoost, LightGBM, CatBoost)
 - Time-series aware validation
 - Comprehensive ablation study
 - Publication-ready visualizations

### SECTION 1: SETUP & IMPORTS


In [3]:
# Core Libraries
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import gc
from meteostat import Point, Hourly
from datetime import datetime, timedelta
# Machine Learning
from sklearn.model_selection import train_test_split, TimeSeriesSplit, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error
from sklearn.feature_selection import mutual_info_regression

# Gradient Boosting Models
from xgboost import XGBRegressor
import lightgbm as lgb
from catboost import CatBoostRegressor

# Time Series
from pandas.tseries.holiday import USFederalHolidayCalendar
import holidays

# Settings
warnings.filterwarnings('ignore')
pd.set_option("display.max_columns", None)
plt.style.use('ggplot')
sns.set_palette("husl")

# Configuration
CONFIG = {
    'base_path': '/kaggle/input/capital-bikeshare-dataset-202005202408/',
    'test_size': 0.2,
    'random_state': 42,
    'cv_folds': 5,
    'include_pandamic':False
}

print(" Libraries loaded successfully!")
print(f" Configuration: {CONFIG}")

ModuleNotFoundError: No module named 'meteostat'

 ##  SECTION 2: DATA LOADING & INITIAL EXPLORATION
 **Key Decision**: We load only the `started_at` column initially to reduce memory usage, then aggregate to hourly level.


In [ ]:
def load_bikeshare_data(base_path):
    """
    Load and perform initial processing of bikeshare data.
    Only loads essential columns for memory efficiency.
    """
    print("\n" + "="*70)
    print("LOADING BIKESHARE DATA")
    print("="*70)
    
    # Find files
    files = [f for f in os.listdir(base_path) if f.endswith('.csv')]
    file_rides = next((f for f in files if 'daily_rent_detail' in f), None)
    
    if not file_rides:
        raise FileNotFoundError(f"Rides file not found in {base_path}")
    
    print(f" Loading: {file_rides}")
    
    # Load only timestamp column (memory efficient)
    df_rides = pd.read_csv(
        os.path.join(base_path, file_rides),
        usecols=['started_at'],
        dtype={'started_at': str}
    )
    
    print(f"   Initial shape: {df_rides.shape}")
    
    # Parse dates (clean first 19 characters to handle milliseconds)
    df_rides['started_at'] = pd.to_datetime(
        df_rides['started_at'].str.slice(0, 19),
        errors='coerce'
    )
    
    # Remove parsing failures
    before_drop = len(df_rides)
    df_rides = df_rides.dropna(subset=['started_at'])
    print(f"   Dropped {before_drop - len(df_rides)} rows with invalid dates")
    
    return df_rides

df_rides = load_bikeshare_data(CONFIG['base_path'])
print(f"\n Date range: {df_rides['started_at'].min()} → {df_rides['started_at'].max()}")


##  SECTION 3: COVID-ERA DATA REMOVAL (CRITICAL)

**Scientific Justification**:
- 2020-2021 saw unprecedented lockdowns and behavior changes
- This represents **concept drift** - the relationship between features and target changed fundamentally
- Including COVID data would confuse the model with non-recurring patterns
- Post-2022 data represents "normal" operations that generalize to future predictions

**Impact**: This is one of the most important preprocessing steps for model performance.


In [ ]:

# %%
print("\n" + "="*70)
print("COVID-ERA DATA REMOVAL")
print("="*70)

# Analyze years
year_counts = df_rides['started_at'].dt.year.value_counts().sort_index()
print("\n Rides by Year (Before Filtering):")
print(year_counts)


if (CONFIG['include_pandamic']):
    #pandamic var
    df_rides['normal_flag'] = (~df_rides['started_at'].dt.year.isin([2020, 2021])).astype(int)
else:
    # Remove 2020-2021
    rows_before = len(df_rides)
    df_rides = df_rides[df_rides['started_at'].dt.year >= 2022]
    rows_after = len(df_rides)

print(f"\n  Removed: {rows_before - rows_after:,} rows from COVID era (2020-2021)")
print(f" Retained: {rows_after:,} rows from normal period (2022+)")
print(f" Data reduction: {(1 - rows_after/rows_before)*100:.1f}%")


##  SECTION 4: HOURLY AGGREGATION

**Why Hourly?**
- Individual rides contain too much noise for prediction
- Hourly aggregation captures demand patterns while maintaining temporal resolution
- Suitable for operational planning (station rebalancing, maintenance scheduling)


In [ ]:

print("\n" + "="*70)
print("HOURLY AGGREGATION")
print("="*70)

# Create hourly timestamp (floor to hour)
df_rides['datetime_hourly'] = df_rides['started_at'].dt.floor('h')

# Aggregate to hourly counts
df_hourly = df_rides.groupby('datetime_hourly').size().reset_index(name='count')
df_hourly = df_hourly.sort_values('datetime_hourly').reset_index(drop=True)

print(f" Aggregated to {len(df_hourly):,} hourly records")
print(f" Average hourly rides: {df_hourly['count'].mean():.1f}")
print(f" Hourly ride range: {df_hourly['count'].min()} to {df_hourly['count'].max()}")

# Free memory
del df_rides
gc.collect()

##  SECTION 5: WEATHER DATA LOADING & LEAKAGE PREVENTION

**Critical Issue - Data Leakage**:
- Raw weather data is "same-day" data
- In production, we predict BEFORE the hour occurs
- We don't know today's weather when predicting today's demand

**Solution**:
- Shift weather data by +1 day
- Use "yesterday's weather" to predict "today's demand"
- This simulates using weather forecasts (which are based on recent conditions)


In [ ]:
# --- REVISED SECTION 5: LOAD WEATHER (API) WITH ORIGINAL COLUMN NAMES ---


def load_weather_compatible():
    print("\n" + "="*70)
    print("WEATHER DATA LOADING (API) & LEAKAGE PREVENTION")
    print("="*70)

    # 1. Define Location (DC) and Dates (2021-2024)
    dc_location = Point(38.9072, -77.0369)
    start = datetime(2021, 1, 1)
    end = datetime(2024, 12, 31)

    print("1. Fetching hourly weather data from Meteostat...")
    data = Hourly(dc_location, start, end)
    df_weather = data.fetch().reset_index()

    # 2. RENAME columns to match your ORIGINAL dataset
    # This ensures your EDA and Model code (which looks for 'temp', 'humidity') works perfectly.
    df_weather = df_weather.rename(columns={
        'time': 'datetime',     # Matches your original 'datetime'
        'temp': 'temp',         # Matches your original 'temp'
        'rhum': 'humidity',     # Matches your original 'humidity'
        'prcp': 'precip',       # Matches your original 'precip'
        'wspd': 'windspeed'     # Matches your original 'windspeed'
    })

    # Note: 'cloudcover' is not available in the basic API response, 
    # so we select only the columns we have to avoid KeyErrors.
    available_cols = ['datetime', 'temp', 'humidity', 'precip', 'windspeed']
    df_weather = df_weather[available_cols]

    # 3. LEAKAGE PREVENTION (The Correct Implementation)
    # Problem in old code: dt.normalize() only removed the time, it didn't shift the day.
    # New Fix: We explicitly add 24 hours.
    # Meaning: Weather recorded on Jan 1st is labeled as Jan 2nd, simulating a 24h forecast.
    #to_update
    # df_weather['datetime'] = df_weather['datetime'] + timedelta(hours=24)
    
    # 4. Create the join keys your model expects
    df_weather['date_key'] = df_weather['datetime'].dt.normalize()
    df_weather['hour'] = df_weather['datetime'].dt.hour
    
    print(f"2. Applied +24h time shift.")
    print(f"   (Real weather from Jan 1 is now available for Jan 2 predictions)")
    print(f"3. Final Columns: {list(df_weather.columns)}")
    
    return df_weather

# Load the data
df_weather = load_weather_compatible()



In [ ]:
df_weather.shape

##  SECTION 6: MERGE RIDES & WEATHER

We merge on the date (normalized to midnight) to assign daily weather to each hour.


In [ ]:
print("\n" + "="*70)
print("MERGING RIDES & WEATHER (CORRECTED)")
print("="*70)

# 1. Prepare Join Keys for Ride Data
df_hourly['date_key'] = df_hourly['datetime_hourly'].dt.normalize()
df_hourly['hour'] = df_hourly['datetime_hourly'].dt.hour  # <--- NEW: You need this for the merge

# 2. Merge on BOTH Date and Hour
# This ensures 1-to-1 matching (8 AM rides matched ONLY with 8 AM weather)
df = pd.merge(df_hourly, df_weather, on=['date_key', 'hour'], how='inner')

print(f" Merged shape: {df.shape}")
print(f" Final date range: {df['datetime_hourly'].min()} → {df['datetime_hourly'].max()}")
print(f" Total hours of data: {len(df):,}")

# Clean up
df = df.drop(columns=['date_key', 'datetime'], errors='ignore')

# Verify the fix (Should be ~23k, not 560k)
if len(df) > 30000:
    print(" WARNING: Dataset is still too large! Check for duplicates in df_weather.")
else:
    print(" Success: Dataset size is correct.")

##  SECTION 7: EXPLORATORY DATA ANALYSIS (EDA)

Understanding the data before modeling is crucial for feature engineering decisions.


In [ ]:


print("\n" + "="*70)
print("EXPLORATORY DATA ANALYSIS")
print("="*70)

# Basic statistics
print(f"\n Target Variable Statistics (count):")
print(df['count'].describe())
print(f"\n  Skewness: {df['count'].skew():.2f} (Right-skewed → Log transform recommended)")

# Create temporal features for EDA
df['hour'] = df['datetime_hourly'].dt.hour
df['day_of_week'] = df['datetime_hourly'].dt.dayofweek
df['month'] = df['datetime_hourly'].dt.month
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

# Visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Exploratory Data Analysis - Bikeshare Demand Patterns', fontsize=16, fontweight='bold')

# 1. Hourly Pattern (Most Important)
hourly_avg = df.groupby('hour')['count'].mean()
axes[0, 0].bar(hourly_avg.index, hourly_avg.values, color='steelblue', edgecolor='black')
axes[0, 0].axvline(8, color='red', linestyle='--', alpha=0.7, linewidth=2, label='Morning Rush')
axes[0, 0].axvline(17, color='red', linestyle='--', alpha=0.7, linewidth=2, label='Evening Rush')
axes[0, 0].set_title('Hourly Demand Pattern (Clear Rush Hours)', fontweight='bold')
axes[0, 0].set_xlabel('Hour of Day')
axes[0, 0].set_ylabel('Average Bike Count')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Weekday vs Weekend
daily_avg = df.groupby('day_of_week')['count'].mean()
day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
colors = ['coral']*5 + ['lightblue']*2
axes[0, 1].bar(day_labels, daily_avg.values, color=colors, edgecolor='black')
axes[0, 1].set_title('Weekday vs Weekend Demand', fontweight='bold')
axes[0, 1].set_ylabel('Average Bike Count')
axes[0, 1].grid(True, alpha=0.3, axis='y')

# 3. Monthly Seasonality
monthly_avg = df.groupby('month')['count'].mean()
axes[0, 2].plot(monthly_avg.index, monthly_avg.values, marker='o', linewidth=2, markersize=8, color='green')
axes[0, 2].set_title('Seasonal Pattern (Monthly)', fontweight='bold')
axes[0, 2].set_xlabel('Month')
axes[0, 2].set_ylabel('Average Bike Count')
axes[0, 2].set_xticks(range(1, 13))
axes[0, 2].grid(True, alpha=0.3)

# 4. Temperature Impact
axes[1, 0].scatter(df['temp'], df['count'], alpha=0.2, s=10, color='orange')
axes[1, 0].set_title('Temperature vs Demand (Positive Correlation)', fontweight='bold')
axes[1, 0].set_xlabel('Temperature (°F)')
axes[1, 0].set_ylabel('Bike Count')
axes[1, 0].grid(True, alpha=0.3)

# 5. Distribution (Skewed)
axes[1, 1].hist(df['count'], bins=50, edgecolor='black', color='purple', alpha=0.7)
axes[1, 1].axvline(df['count'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df["count"].mean():.0f}')
axes[1, 1].set_title('Demand Distribution (Right-Skewed)', fontweight='bold')
axes[1, 1].set_xlabel('Bike Count')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3, axis='y')

# 6. Time Series
sample_df = df.head(24*7)  # First week
axes[1, 2].plot(range(len(sample_df)), sample_df['count'], linewidth=1, color='darkblue')
axes[1, 2].set_title('Sample Time Series (First Week)', fontweight='bold')
axes[1, 2].set_xlabel('Hour Index')
axes[1, 2].set_ylabel('Bike Count')
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n Key Insights:")
print(f"   1. Clear bimodal pattern with rush hours at 8 AM and 5 PM")
print(f"   2. Weekday demand higher than weekend (commuting vs leisure)")
print(f"   3. Strong seasonality: peak in summer months")
print(f"   4. Positive correlation with temperature")
print(f"   5. Right-skewed distribution → log transform needed")


##  SECTION 8: FEATURE ENGINEERING

This is the most critical section. Good features = Good model.

**Feature Categories**:
1. **Temporal**: Hour, day, month with cyclical encoding
2. **Calendar**: Weekends, holidays, rush hours
3. **Weather**: Temperature, precipitation, wind
4. **Interactions**: Weather × Time effects
5. **Lag Features**: Past demand (most important for time series)
6. **Rolling Statistics**: Moving averages and volatility

In [ ]:

print("\n" + "="*70)
print("FEATURE ENGINEERING")
print("="*70)

df_features = df.copy()
# --- HELPER FUNCTIONS ---

def get_solar_features(df, lat=38.9072, lon=-77.0369): 
    """Calculates solar position. Requires 'hour' and 'day_of_year'."""
    # 1. Day of Year
    df['doy'] = df['datetime_hourly'].dt.dayofyear
    
    # 2. Solar Declination (Angle of sun vs equator)
    df['solar_declination'] = -23.44 * np.cos(np.radians(360/365 * (df['doy'] + 10)))
    
    # 3. Hour Angle (Time expressed as angle)
    df['hour_angle'] = 15 * (df['hour'] - 12)
    
    # 4. Solar Elevation (The critical light feature)
    lat_rad = np.radians(lat)
    decl_rad = np.radians(df['solar_declination'])
    ha_rad = np.radians(df['hour_angle'])
    
    df['solar_elevation'] = np.degrees(np.arcsin(
        np.sin(lat_rad) * np.sin(decl_rad) + 
        np.cos(lat_rad) * np.cos(decl_rad) * np.cos(ha_rad)
    ))
    
    # 5. Daylight (Civil Twilight > -6 degrees)
    df['is_daylight'] = (df['solar_elevation'] > -6).astype(int)
    return df

def calculate_apparent_temp(df):
    """Real feel temperature using Temp, Wind, Humidity"""
    T = df['temp']
    WS = df['windspeed']
    RH = df['humidity']
    
    # Vapor Pressure approx
    e = (RH / 100) * 6.105 * np.exp((17.27 * T) / (237.7 + T))
    
    # Australian BOM formula
    df['apparent_temp'] = T + (0.33 * e) - (0.70 * WS) - 4.00
    return df
def add_fourier_terms(df, period=24*365, K=3):
    """
    Adds Fourier terms for annual seasonality.
    K: Number of sine/cosine pairs (Higher K = more complex curve)
    """
    t = df['datetime_hourly'].astype(np.int64) // 10**9 # Timestamp in seconds
    
    for k in range(1, K+1):
        df[f'sin_annual_{k}'] = np.sin(2 * np.pi * k * t / period)
        df[f'cos_annual_{k}'] = np.cos(2 * np.pi * k * t / period)
        
    return df
# ==========================================
# 1. TEMPORAL FEATURES (Already created in EDA)
# ==========================================
print("\n1️ Temporal features already created in EDA")

# ==========================================
# 2. CYCLICAL ENCODING (CRITICAL FOR PERIODIC FEATURES)
# ==========================================
print("2️ Creating cyclical encoding (sine/cosine transforms)...")
print("   Why? Hours 0 and 23 are adjacent, but 23-0=23 numerically. Sin/cos preserves circularity.")
#to_update
# # Hour (24-hour cycle)
df_features['hour_sin'] = np.sin(2 * np.pi * df_features['hour'] / 24)
df_features['hour_cos'] = np.cos(2 * np.pi * df_features['hour'] / 24)

# Month (12-month cycle)
df_features['month_sin'] = np.sin(2 * np.pi * df_features['month'] / 12)
df_features['month_cos'] = np.cos(2 * np.pi * df_features['month'] / 12)

# Day of week (7-day cycle)
df_features['dow_sin'] = np.sin(2 * np.pi * df_features['day_of_week'] / 7)
df_features['dow_cos'] = np.cos(2 * np.pi * df_features['day_of_week'] / 7)


# -Solar and feeling features
print("2b. Adding Solar & Physics Features...")
df_features = get_solar_features(df_features)
df_features = calculate_apparent_temp(df_features)


# ==========================================
# 3. CALENDAR FEATURES
# ==========================================
print("3️ Creating calendar features (holidays, rush hours)...")

# US Federal Holidays
cal = USFederalHolidayCalendar()
holidays_dates = cal.holidays(
    start=df_features['datetime_hourly'].min(),
    end=df_features['datetime_hourly'].max()
)
df_features['is_holiday'] = df_features['datetime_hourly'].dt.date.isin(
    holidays_dates.date
).astype(int)

# Working Day (Consolidated)
df_features['is_working_day'] = (
    (df_features['is_weekend'] == 0) & 
    (df_features['is_holiday'] == 0)
).astype(int)


# Rush Hour (Domain Knowledge)
def identify_rush_hour(row):
    """
    Rush hours: 7-9 AM and 4-7 PM on weekdays (not holidays)
    This captures commuter behavior
    """
    if row['is_weekend'] == 1 or row['is_holiday'] == 1:
        return 0
    if (7 <= row['hour'] <= 9) or (16 <= row['hour'] <= 19):
        return 1
    return 0

df_features['is_rush_hour'] = df_features.apply(identify_rush_hour, axis=1)

print(f"   Holiday hours: {df_features['is_holiday'].sum():,}")
print(f"   Rush hours: {df_features['is_rush_hour'].sum():,}")

# ==========================================
# 4. WEATHER PROCESSING
# ==========================================
print("4️ Processing weather features...")

# Handle missing values in weather
weather_features = ['temp', 'humidity', 'precip', 'windspeed', 'cloudcover']
for col in weather_features:
    if col in df_features.columns:
        # Interpolate then fill remaining with median
        df_features[col] = df_features[col].interpolate(method='linear')
        df_features[col] = df_features[col].fillna(df_features[col].median())

# Binary rain indicator (threshold: 0.1 inches)
if 'precip' in df_features.columns:
    df_features['is_raining'] = (df_features['precip'] > 0.1).astype(int)
    print(f"   Rainy hours: {df_features['is_raining'].sum():,} ({df_features['is_raining'].mean()*100:.1f}%)")
# --- SECTION 8: FEATURE ENGINEERING (ADD SYNTHETIC WEATHER) ---

# 1. Drop the empty 'snow_depth' column if it exists
if 'snow_depth' in df_features.columns:
    df_features = df_features.drop(columns=['snow_depth'])

# 2. Create Synthetic "Is Snowing" Feature
# Logic: Precip > 0 (it's wet) AND Temp <= 2°C (it's freezing)
# We use 2°C (35.6°F) because snow often mixes with rain slightly above freezing.
df_features['is_snowing'] = (
    (df_features['precip'] > 0) & 
    (df_features['temp'] <= 2)
).astype(int)

# 4. Create "Extreme Weather" Flag
# High winds (e.g., > 30 km/h) + Precip is very bad for biking
df_features['is_bad_weather'] = (
    (df_features['precip'] > 0) | (df_features['windspeed'] > 35)
).astype(int)

print("Created synthetic weather features:")
print(f"  Snowing Hours: {df_features['is_snowing'].sum()}")
print(f"  Raining Hours: {df_features['is_raining'].sum()}")

# ==========================================
# 5. INTERACTION FEATURES
# ==========================================
print("5️ Creating interaction features...")
print("   Why? Effects are not always additive. Good weather has MORE impact during rush hour.")

if 'temp' in df_features.columns:
    df_features['temp_x_hour'] = df_features['temp'] * df_features['hour']
    df_features['temp_x_weekend'] = df_features['temp'] * df_features['is_weekend']

if 'precip' in df_features.columns and 'is_rush_hour' in df_features.columns:
    df_features['rain_x_rushhour'] = df_features['precip'] * df_features['is_rush_hour']

if 'humidity' in df_features.columns:
    df_features['humidity_x_temp'] = df_features['humidity'] * df_features['temp']

# ==========================================
# 6. LAG FEATURES (MOST IMPORTANT FOR TIME SERIES)
# ==========================================
print("6️ Creating lag features (time series memory)...")
print("   Why? Past demand is the strongest predictor of future demand!")

# Sort by time (critical for lags)
df_features = df_features.sort_values('datetime_hourly').reset_index(drop=True)

# Create target log transform BEFORE creating lags
df_features['count_log'] = np.log1p(df_features['count'])

# Lag features on log-transformed target
df_features['lag_1h'] = df_features['count_log'].shift(1)       # 1 hour ago
df_features['lag_24h'] = df_features['count_log'].shift(24)     # Same hour yesterday  
df_features['lag_168h'] = df_features['count_log'].shift(168)   # Same hour last week

# ==========================================
# 7. ROLLING STATISTICS
# ==========================================
print("7️ Creating rolling statistics...")
print("   Why? Captures recent trends and volatility")

# Rolling means (short and medium term)
df_features['rolling_mean_3h'] = df_features['count_log'].shift(1).rolling(3).mean()
df_features['rolling_mean_12h'] = df_features['count_log'].shift(1).rolling(12).mean()
df_features['rolling_mean_24h'] = df_features['count_log'].shift(1).rolling(24).mean()



# ==========================================
# 8. CLEANUP
# ==========================================
print("8️ Cleaning up NaN values from lag/rolling features...")

rows_before = len(df_features)
df_features = df_features.dropna()
rows_after = len(df_features)

print(f"   Dropped {rows_before - rows_after:,} rows with NaN (from shift operations)")
print(f"   Final dataset: {rows_after:,} rows")

print(f"\n Feature engineering complete! Total features: {len(df_features.columns)}")


##  SECTION 9: FEATURE CORRELATION ANALYSIS

Understanding feature relationships helps with feature selection and model interpretation.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

print("\n" + "="*70)
print("FEATURE CORRELATION ANALYSIS")
print("="*70)

# Select numeric features for correlation
numeric_features = df_features.select_dtypes(include=[np.number]).columns.tolist()
numeric_features = [f for f in numeric_features if f not in ['count', 'datetime_hourly']]

# Calculate correlation with target
target_correlations = df_features[numeric_features].corrwith(df_features['count_log']).sort_values(ascending=False)

print("\n Top 15 Features Correlated with Target (count_log):")
print(target_correlations.head(30))

# --- Figure 1: Top correlations ---
top_n = 30
plt.figure(figsize=(16, 6))
plt.barh(range(top_n), target_correlations.head(top_n).values, color='green', alpha=0.7)
plt.yticks(range(top_n), target_correlations.head(top_n).index)
plt.xlabel('Correlation Coefficient')
plt.title('Top 30 Positively Correlated Features', fontweight='bold')
plt.grid(True, alpha=0.3, axis='x')
plt.gca().invert_yaxis()
plt.show()

# --- Figure 2: Correlation heatmap ---
key_features = ['count_log', 'lag_1h', 'lag_24h', 'lag_168h', 'rolling_mean_24h',
                'hour_sin', 'hour_cos', 'temp', 'is_rush_hour', 'is_weekend', 
                'precip', 'humidity', 'windspeed']

# Keep only features present in the DataFrame
key_features = [f for f in key_features if f in df_features.columns]

plt.figure(figsize=(12, 10))
corr_matrix = df_features[key_features].corr()
sns.heatmap(corr_matrix, annot=False, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlation Heatmap ', fontweight='bold')
plt.show()

print("\nKey Observations:")
print("   1. Lag features (lag_1h, lag_24h) have highest correlation → Time series memory is crucial")
print("   2. Rolling averages capture trend information")
print("   3. Rush hour and temporal features show moderate correlation")
print("   4. Weather features have weaker but meaningful correlation")


##  SECTION 10: PREPARE DATA FOR MODELING

Split data into features (X) and target (y), then create train/test sets.


In [ ]:

print("\n" + "="*70)
print("PREPARING DATA FOR MODELING")
print("="*70)

# Define feature list (exclude target and metadata)
exclude_cols = ['datetime_hourly', 'count', 'count_log', 'date_key']
feature_columns = [col for col in df_features.columns if col not in exclude_cols]

print(f" Total features: {len(feature_columns)}")
print(f" Feature list: {feature_columns}")

# Prepare X and y
X = df_features[feature_columns]
y = df_features['count_log']  # Use log-transformed target

# Time-based split (critical for time series)
# We DON'T shuffle because order matters!
split_point = int(len(X) * (1 - CONFIG['test_size']))
X_train, X_test = X.iloc[:split_point], X.iloc[split_point:]
y_train, y_test = y.iloc[:split_point], y.iloc[split_point:]

print(f"\n Train set: {len(X_train):,} samples ({(1-CONFIG['test_size'])*100:.0f}%)")
print(f" Test set:  {len(X_test):,} samples ({CONFIG['test_size']*100:.0f}%)")
print(f" Train period: {df_features.iloc[:split_point]['datetime_hourly'].min()} → "
      f"{df_features.iloc[:split_point]['datetime_hourly'].max()}")
print(f" Test period:  {df_features.iloc[split_point:]['datetime_hourly'].min()} → "
      f"{df_features.iloc[split_point:]['datetime_hourly'].max()}")

##  SECTION 11: FEATURE SCALING (For Linear Models)

Tree-based models (RF, XGBoost, LightGBM, CatBoost) don't need scaling.
But Linear Regression benefits from standardization.


In [ ]:

print("\n" + "="*70)
print("FEATURE SCALING")
print("="*70)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(" Features scaled using StandardScaler (zero mean, unit variance)")
print("   Note: Tree-based models will use unscaled data")


##  SECTION 12: MODEL TRAINING & EVALUATION

We'll train 5 models and compare performance:
1. Linear Regression (baseline)
2. Random Forest
3. XGBoost
4. LightGBM
5. CatBoost

In [ ]:

def evaluate_model(model, X_test, y_test_log, model_name):
    """
    Comprehensive model evaluation with metrics on original scale.
    
    Returns both log-scale and real-scale metrics.
    """
    # Predictions (log scale)
    y_pred_log = model.predict(X_test)
    
    # Transform back to original scale
    y_test_real = np.expm1(y_test_log)
    y_pred_real = np.expm1(y_pred_log)
    
    # Metrics on log scale
    r2_log = r2_score(y_test_log, y_pred_log)
    rmse_log = np.sqrt(mean_squared_error(y_test_log, y_pred_log))
    
    # Metrics on real scale
    r2_real = r2_score(y_test_real, y_pred_real)
    rmse_real = np.sqrt(mean_squared_error(y_test_real, y_pred_real))
    mae_real = mean_absolute_error(y_test_real, y_pred_real)
    mape_real = mean_absolute_percentage_error(y_test_real, y_pred_real) * 100
    
    # Adjusted R²
    n, p = len(y_test_real), X_test.shape[1]
    adj_r2 = 1 - (1 - r2_real) * (n - 1) / (n - p - 1)
    
    print(f"\n{'='*70}")
    print(f"{model_name.upper()} - EVALUATION RESULTS")
    print(f"{'='*70}")
    print(f" Log Scale Metrics:")
    print(f"   R² Score: {r2_log:.4f}")
    print(f"   RMSE:     {rmse_log:.4f}")
    print(f"\n Real Scale Metrics (Bike Counts):")
    print(f"   R² Score:     {r2_real:.4f}")
    print(f"   Adjusted R²:  {adj_r2:.4f}")
    print(f"   RMSE:         {rmse_real:.2f} bikes")
    print(f"   MAE:          {mae_real:.2f} bikes")
    print(f"   MAPE:         {mape_real:.2f}%")
    print(f"{'='*70}")
    
    return {
        'Model': model_name,
        'R2_log': r2_log,
        'R2_real': r2_real,
        'Adj_R2': adj_r2,
        'RMSE_real': rmse_real,
        'MAE_real': mae_real,
        'MAPE': mape_real,
        'predictions_log': y_pred_log,
        'predictions_real': y_pred_real
    }

# %%
print("\n" + "="*70)
print("MODEL 1: LINEAR REGRESSION (Baseline)")
print("="*70)

lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

results_lr = evaluate_model(lr_model, X_test_scaled, y_test, "Linear Regression")

# %%
print("\n" + "="*70)
print("MODEL 2: RANDOM FOREST")
print("="*70)

rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=CONFIG['random_state'],
    n_jobs=-1,
    verbose=0
)

print("Training Random Forest...")
rf_model.fit(X_train, y_train)

results_rf = evaluate_model(rf_model, X_test, y_test, "Random Forest")

# %%
print("\n" + "="*70)
print("MODEL 3: XGBOOST")
print("="*70)

xgb_model = XGBRegressor(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=CONFIG['random_state'],
    n_jobs=-1
)

print("Training XGBoost...")
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

results_xgb = evaluate_model(xgb_model, X_test, y_test, "XGBoost")

# %%
print("\n" + "="*70)
print("MODEL 4: LIGHTGBM")
print("="*70)

lgb_model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=10,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=CONFIG['random_state'],
    n_jobs=-1,
    verbose=-1,
    objective='tweedie',           # Better for count data with zeros
    tweedie_variance_power=1.1,    # 1.0=Poisson, 2.0=Gamma. 1.1 is best for demand.
    metric='rmse'                  # We still optimize for RMSE
)

print("Training LightGBM...")
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
)

results_lgb = evaluate_model(lgb_model, X_test, y_test, "LightGBM")

# %%
print("\n" + "="*70)
print("MODEL 5: CATBOOST")
print("="*70)

cat_model = CatBoostRegressor(
    iterations=2000,
    learning_rate=0.03,
    depth=6,
    loss_function='RMSE',
    random_seed=CONFIG['random_state'],
    verbose=0
)

print("Training CatBoost...")
cat_model.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
    early_stopping_rounds=50,
    verbose=False
)

results_cat = evaluate_model(cat_model, X_test, y_test, "CatBoost")


##  SECTION 13: MODEL COMPARISON & VISUALIZATION


In [ ]:

print("\n" + "="*70)
print("MODEL COMPARISON SUMMARY")
print("="*70)

# Compile results
comparison_df = pd.DataFrame([
    {
        'Model': results_lr['Model'],
        'R²': results_lr['R2_real'],
        'Adj R²': results_lr['Adj_R2'],
        'RMSE': results_lr['RMSE_real'],
        'MAE': results_lr['MAE_real'],
        'MAPE (%)': results_lr['MAPE']
    },
    {
        'Model': results_rf['Model'],
        'R²': results_rf['R2_real'],
        'Adj R²': results_rf['Adj_R2'],
        'RMSE': results_rf['RMSE_real'],
        'MAE': results_rf['MAE_real'],
        'MAPE (%)': results_rf['MAPE']
    },
    {
        'Model': results_xgb['Model'],
        'R²': results_xgb['R2_real'],
        'Adj R²': results_xgb['Adj_R2'],
        'RMSE': results_xgb['RMSE_real'],
        'MAE': results_xgb['MAE_real'],
        'MAPE (%)': results_xgb['MAPE']
    },
    {
        'Model': results_lgb['Model'],
        'R²': results_lgb['R2_real'],
        'Adj R²': results_lgb['Adj_R2'],
        'RMSE': results_lgb['RMSE_real'],
        'MAE': results_lgb['MAE_real'],
        'MAPE (%)': results_lgb['MAPE']
    },
    {
        'Model': results_cat['Model'],
        'R²': results_cat['R2_real'],
        'Adj R²': results_cat['Adj_R2'],
        'RMSE': results_cat['RMSE_real'],
        'MAE': results_cat['MAE_real'],
        'MAPE (%)': results_cat['MAPE']
    }
])

print(comparison_df.to_string(index=False))

# Find best model
best_model_name = comparison_df.loc[comparison_df['R²'].idxmax(), 'Model']
print(f"\n Best Model: {best_model_name}")

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Model Performance Comparison', fontsize=16, fontweight='bold')

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

# 1. R² Comparison
axes[0, 0].barh(comparison_df['Model'], comparison_df['R²'], color=colors)
axes[0, 0].set_xlabel('R² Score (Higher is Better)')
axes[0, 0].set_title('R² Score Comparison')
axes[0, 0].axvline(0.9, color='red', linestyle='--', alpha=0.5, label='Excellent (0.9)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3, axis='x')

# 2. RMSE Comparison
axes[0, 1].barh(comparison_df['Model'], comparison_df['RMSE'], color=colors)
axes[0, 1].set_xlabel('RMSE (Lower is Better)')
axes[0, 1].set_title('RMSE Comparison')
axes[0, 1].grid(True, alpha=0.3, axis='x')

# 3. MAE Comparison
axes[1, 0].barh(comparison_df['Model'], comparison_df['MAE'], color=colors)
axes[1, 0].set_xlabel('MAE (Lower is Better)')
axes[1, 0].set_title('Mean Absolute Error Comparison')
axes[1, 0].grid(True, alpha=0.3, axis='x')

# 4. MAPE Comparison
axes[1, 1].barh(comparison_df['Model'], comparison_df['MAPE (%)'], color=colors)
axes[1, 1].set_xlabel('MAPE % (Lower is Better)')
axes[1, 1].set_title('Mean Absolute Percentage Error')
axes[1, 1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()


##  SECTION 14: TIME SERIES CROSS-VALIDATION

**Why TimeSeriesSplit?**
- Standard K-Fold shuffles data → breaks temporal order → data leakage
- TimeSeriesSplit respects chronological order
- Each fold uses past data to predict future data


In [ ]:

print("\n" + "="*70)
print("TIME SERIES CROSS-VALIDATION")
print("="*70)

tscv = TimeSeriesSplit(n_splits=CONFIG['cv_folds'])

print(f"Using {CONFIG['cv_folds']}-fold Time Series Cross-Validation")
print("Training chronologically: Fold 1 < Fold 2 < Fold 3 < ...\n")

# Models to validate (excluding Linear Regression for speed)
models_cv = {
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=CONFIG['random_state'], n_jobs=-1),
    'XGBoost': XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=CONFIG['random_state'], n_jobs=-1),
    'LightGBM': lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, random_state=CONFIG['random_state'], n_jobs=-1, verbose=-1),
    'CatBoost': CatBoostRegressor(iterations=300, learning_rate=0.05, random_seed=CONFIG['random_state'], verbose=0)
}

cv_results = []

for model_name, model in models_cv.items():
    print(f"Validating {model_name}...")
    
    fold_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train)):
        # Split fold
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        # Train
        if model_name == 'CatBoost':
            model.fit(X_fold_train, y_fold_train, verbose=False)
        else:
            model.fit(X_fold_train, y_fold_train)
        
        # Predict and evaluate
        y_fold_pred = model.predict(X_fold_val)
        fold_rmse = np.sqrt(mean_squared_error(y_fold_val, y_fold_pred))
        fold_scores.append(fold_rmse)
    
    mean_score = np.mean(fold_scores)
    std_score = np.std(fold_scores)
    
    print(f"   {model_name:15s}: CV RMSE = {mean_score:.4f} ± {std_score:.4f} (log scale)")
    
    cv_results.append({
        'Model': model_name,
        'CV_RMSE': mean_score,
        'CV_Std': std_score
    })

cv_results_df = pd.DataFrame(cv_results)
print(f"\n{'='*70}")
print("CROSS-VALIDATION SUMMARY")
print(f"{'='*70}")
print(cv_results_df.to_string(index=False))


## CrossValidation Best Paramas takes time do not run it (30min) waaaaaaaaaaaaaaaaaaaaaaaaa

In [ ]:
# from sklearn.model_selection import RandomizedSearchCV
# from catboost import CatBoostRegressor

# print("\n" + "="*70)
# print("HYPERPARAMETER TUNING (CatBoost on Dual-GPU 🚀🚀)")
# print("="*70)

# # 1. Define the CV with a GAP
# tscv_gap = TimeSeriesSplit(n_splits=5, gap=24) 

# # 2. Define the Model with DUAL GPU Support
# cat_base = CatBoostRegressor(
#     loss_function='RMSE',
#     task_type="GPU",
    
#     # --- DUAL GPU SETTING ---
#     devices='0:1',               # Use GPU 0 AND GPU 1 together
#     # ------------------------
    
#     bootstrap_type='Bernoulli',  # Required for subsample on GPU
#     verbose=0,
#     random_seed=42,
#     allow_writing_files=False
# )

# # 3. Define the Search Space
# param_dist = {
#     'iterations': [1000, 2000, 3000],        
#     'learning_rate': [0.01, 0.03, 0.05, 0.1],
#     'depth': [6, 8, 10],                     
#     'l2_leaf_reg': [1, 3, 5, 9],
#     'subsample': [0.6, 0.8, 0.95],
#     'min_data_in_leaf': [1, 10, 20]
# }

# # 4. Run Randomized Search
# print("⏳ Tuning CatBoost parameters on 2x GPUs (Sequential Mode)...")
# search = RandomizedSearchCV(
#     estimator=cat_base,
#     param_distributions=param_dist,
#     n_iter=20,
#     scoring='neg_root_mean_squared_error',
#     cv=tscv_gap,
    
#     # --- CRITICAL ---
#     n_jobs=1,  # Run 1 job at a time. That job will use BOTH GPUs.
#     # ----------------
    
#     verbose=1,
#     random_state=42,
#     error_score='raise'
# )

# try:
#     search.fit(X_train, y_train)
    
#     # 5. Output Results
#     print(f"\n✅ Best Parameters Found: {search.best_params_}")
#     print(f"✅ Best CV RMSE: {-search.best_score_:.4f}")

#     # 6. Save the best model
#     best_model = search.best_estimator_

# except Exception as e:
#     print(f"\n❌ Tuning Failed: {str(e)}")

##  SECTION 15: FEATURE IMPORTANCE ANALYSIS

Understanding which features drive predictions is crucial for model interpretation.


In [ ]:

print("\n" + "="*70)
print("FEATURE IMPORTANCE ANALYSIS")
print("="*70)

# Use best tree-based model 
best_model = cat_model  # Or cat_model based on results

# Get feature importance
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\n Top 20 Most Important Features:")
print(feature_importance.head(20).to_string(index=False))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Top 20 features
top_20 = feature_importance.head(20)
axes[0].barh(range(len(top_20)), top_20['importance'], color='teal')
axes[0].set_yticks(range(len(top_20)))
axes[0].set_yticklabels(top_20['feature'])
axes[0].set_xlabel('Importance (Gain)')
axes[0].set_title('Top 20 Feature Importances (CatBoost)', fontweight='bold')
axes[0].invert_yaxis()
axes[0].grid(True, alpha=0.3, axis='x')

# Feature importance by category
feature_categories = {
    'Lag Features': [f for f in feature_columns if 'lag' in f or 'rolling' in f],
    'Temporal': [f for f in feature_columns if any(x in f for x in ['hour', 'month', 'dow', 'weekend', 'holiday', 'rush'])],
    'Weather': [f for f in feature_columns if any(x in f for x in ['temp', 'precip', 'humidity', 'wind', 'cloud', 'rain'])],
    'Interactions': [f for f in feature_columns if '_x_' in f or 'rain_x' in f]
}

category_importance = {}
for category, features in feature_categories.items():
    features_in_data = [f for f in features if f in feature_importance['feature'].values]
    total_importance = feature_importance[feature_importance['feature'].isin(features_in_data)]['importance'].sum()
    category_importance[category] = total_importance

category_df = pd.DataFrame(list(category_importance.items()), columns=['Category', 'Total Importance'])
category_df = category_df.sort_values('Total Importance', ascending=True)

axes[1].barh(category_df['Category'], category_df['Total Importance'], color=['#d62728', '#2ca02c', '#ff7f0e', '#9467bd'])
axes[1].set_xlabel('Total Importance')
axes[1].set_title('Feature Importance by Category', fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\n Interpretation:")
print("   1. Lag features dominate → Past demand is best predictor")
print("   2. Temporal patterns (hour, rush hour) are second most important")
print("   3. Weather has moderate importance")
print("   4. Interactions capture non-linear effects")


##  SECTION 16: ABLATION STUDY

**Purpose**: Quantify the marginal contribution of each feature group.

We'll remove feature groups one at a time and measure performance degradation.


In [ ]:


print("\n" + "="*70)
print("GRANULAR ABLATION STUDY: INDIVIDUAL FEATURE IMPACT")
print("="*70)

# 0. ENSURE FEATURES ARE DEFINED
# (Using the list from your previous logs)
feature_cols = [
    'hour', 'temp', 'humidity', 'precip', 'windspeed', 'day_of_week', 'month', 'is_weekend', 
    'month_sin', 'month_cos', 'dow_sin', 'dow_cos', 'doy','hour_cos','hour_sin', 
    'solar_declination', 'hour_angle', 'solar_elevation', 'is_daylight', 'apparent_temp', 
    'is_holiday', 'is_working_day', 'is_rush_hour', 
    'is_raining', 'is_snowing', 'is_bad_weather', 
    'temp_x_hour', 'temp_x_weekend', 'rain_x_rushhour', 'humidity_x_temp', 
    'lag_1h', 'lag_24h', 'lag_168h', 
    'rolling_mean_3h', 'rolling_mean_12h', 'rolling_mean_24h'
]

# 1. Define Granular Groups (Separating individual heavy-hitters)
ablation_groups = {
    'Baseline (All Features)': [], 
    
    # -- Time Series Memory --
    ' Remove Lag 1h (Immediate Past)': ['lag_1h'],
    ' Remove Rolling 3h (Recent Trend)': ['rolling_mean_3h'],
    ' Remove Lag 24h (Yesterday)': ['lag_24h', 'rolling_mean_24h'],
    ' Remove Lag 168h (Last Week)': ['lag_168h'],
    
    # -- Weather Specifics --
    ' Remove Temperature': ['temp', 'apparent_temp', 'temp_x_hour', 'temp_x_weekend'],
    ' Remove Rain/Snow Info': ['precip', 'is_raining', 'is_snowing', 'is_bad_weather', 'rain_x_rushhour'],
    ' Remove Humidity': ['humidity', 'humidity_x_temp'],
    
    # -- Time Context --
    ' Remove Hour Info': ['hour', 'is_rush_hour', 'is_daylight', 'solar_elevation'],
    ' Remove Calendar (Month/Day)': ['month', 'day_of_week', 'is_weekend', 'is_holiday']
}

results = []

# 2. Run the Study
for group_name, features_to_drop in ablation_groups.items():
    print(f"Testing: {group_name}...")
    
    # Filter features
    current_features = [f for f in feature_cols if f not in features_to_drop]
    
    # Safety Check
    current_features = [c for c in current_features if c in X_train.columns]
    if not current_features:
        continue
        
    # Train (Fast Mode)
    model_ablation = CatBoostRegressor(
        iterations=300,  # Lower iterations for speed
        learning_rate=0.1, # Slightly higher LR to converge faster
        depth=6,
        loss_function='RMSE',
        verbose=False,
        random_seed=42,
        allow_writing_files=False
    )
    
    model_ablation.fit(X_train[current_features], y_train)
    
    # Predict
    preds_log = model_ablation.predict(X_test[current_features])
    rmse_log = np.sqrt(mean_squared_error(y_test, preds_log))
    
    # Calculate Impact
    if group_name == 'Baseline (All Features)':
        baseline_score = rmse_log
        pct_change = 0.0
    else:
        diff = rmse_log - baseline_score
        pct_change = (diff / baseline_score) * 100

    results.append({
        'Feature Group': group_name,
        'RMSE': rmse_log,
        'Impact (%)': pct_change
    })

# 3. Create DataFrame
df_ablation = pd.DataFrame(results)

# 4. PLOTTING THE RESULTS
plt.figure(figsize=(12, 8))
sns.set_style("whitegrid")

# Create color palette: Red for high impact (bad to remove), Grey for low impact
df_ablation['Color'] = df_ablation['Impact (%)'].apply(lambda x: 'red' if x > 50 else ('orange' if x > 10 else 'grey'))

# Bar Plot
barplot = sns.barplot(
    data=df_ablation[df_ablation['Feature Group'] != 'Baseline (All Features)'],
    y='Feature Group',
    x='Impact (%)',
    palette=df_ablation[df_ablation['Feature Group'] != 'Baseline (All Features)']['Color'].tolist()
)

# Add Baseline Line
plt.axvline(x=0, color='black', linestyle='-', linewidth=1)
plt.title('Feature Ablation Impact: What hurts the model most?', fontsize=15, fontweight='bold')
plt.xlabel('% Increase in Error when Removed (Higher is more important)', fontsize=12)
plt.ylabel('')

# Add value labels
for i, p in enumerate(barplot.patches):
    width = p.get_width()
    plt.text(width + 1, p.get_y() + p.get_height()/2, f'+{width:.1f}%', va='center')

plt.tight_layout()
plt.show()

# 5. Display Table
print(df_ablation[['Feature Group', 'RMSE', 'Impact (%)']].round(4))

In [ ]:
# --- TRAIN "PLANNING MODEL" (NO SHORT-TERM CHEATS) ---

# 1. Define Features (Dropping the "Cheats")
planning_features = [
    'hour', 'month', 'day_of_week', 'is_weekend', 'is_holiday',
    'temp', 'humidity', 'windspeed', 
    'is_raining', 'is_snowing', 'is_bad_weather', 
    'lag_24h', 'lag_168h', 'rolling_mean_24h' # Only Past Data
]

print(f"Training Planning Model with {len(planning_features)} features...")

# 2. Train
model_planning = CatBoostRegressor(
    iterations=1000, 
    learning_rate=0.05, 
    depth=6, 
    loss_function='RMSE',
    verbose=100
)

model_planning.fit(X_train[planning_features], y_train)

# 3. Check Feature Importance
importance = pd.DataFrame({
    'feature': planning_features,
    'importance': model_planning.get_feature_importance()
}).sort_values('importance', ascending=False)

print("\nTop 10 Drivers of Demand (Planning Mode):")
print(importance.head(10))


##  SECTION 17: PREDICTION VISUALIZATION

Visualize actual vs predicted values to assess model performance visually.


In [ ]:


print("\n" + "="*70)
print("PREDICTION VISUALIZATION")
print("="*70)

# Use best model (CatBoost)
y_pred_best_log = results_cat['predictions_log']
y_pred_best_real = results_cat['predictions_real']
y_test_real = np.expm1(y_test)

# Create visualization
fig, axes = plt.subplots(3, 1, figsize=(18, 14))
fig.suptitle('Model Predictions Analysis (Best Model: CatBoost)', fontsize=16, fontweight='bold')

# 1. Time Series Plot (First 500 hours)
n_display = min(500, len(y_test_real))
axes[0].plot(range(n_display), y_test_real.values[:n_display], 
             label='Actual', color='steelblue', linewidth=1.5, alpha=0.8)
axes[0].plot(range(n_display), y_pred_best_real[:n_display], 
             label='Predicted', color='red', linestyle='--', linewidth=1.5, alpha=0.9)
axes[0].set_title(f'Hourly Demand: Actual vs Predicted (First {n_display} Hours)', fontweight='bold')
axes[0].set_xlabel('Hour Index')
axes[0].set_ylabel('Bike Count')
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)

# 2. Scatter Plot (Actual vs Predicted)
axes[1].scatter(y_test_real, y_pred_best_real, alpha=0.3, s=10, color='steelblue')
axes[1].plot([y_test_real.min(), y_test_real.max()], 
             [y_test_real.min(), y_test_real.max()], 
             'r--', linewidth=2, label='Perfect Prediction')
axes[1].set_xlabel('Actual Bike Count')
axes[1].set_ylabel('Predicted Bike Count')
axes[1].set_title('Actual vs Predicted (Scatter Plot)', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 3. Residual Distribution
residuals = y_test_real - y_pred_best_real
axes[2].hist(residuals, bins=80, edgecolor='black', color='purple', alpha=0.7)
axes[2].axvline(0, color='red', linestyle='--', linewidth=2, label='Zero Error')
axes[2].axvline(residuals.mean(), color='green', linestyle='--', linewidth=2, 
                label=f'Mean: {residuals.mean():.2f}')
axes[2].set_xlabel('Residual (Actual - Predicted)')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Residual Distribution (Should be Centered at Zero)', fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Residual statistics
print(f"\n Residual Analysis:")
print(f"   Mean Error:     {residuals.mean():.2f} bikes (should be ~0)")
print(f"   Std Deviation:  {residuals.std():.2f} bikes")
print(f"   Min Error:      {residuals.min():.2f} bikes (underprediction)")
print(f"   Max Error:      {residuals.max():.2f} bikes (overprediction)")
print(f"   Median Error:   {residuals.median():.2f} bikes")


##  SECTION 18: ERROR ANALYSIS BY SEGMENTS

Understand where the model performs well and where it struggles.


In [ ]:

print("\n" + "="*70)
print("ERROR ANALYSIS BY SEGMENTS")
print("="*70)

# Prepare test dataframe with predictions
df_test_analysis = df_features.iloc[split_point:].copy()
df_test_analysis['predicted'] = y_pred_best_real
df_test_analysis['actual'] = y_test_real.values
df_test_analysis['error'] = df_test_analysis['actual'] - df_test_analysis['predicted']
df_test_analysis['abs_error'] = np.abs(df_test_analysis['error'])
df_test_analysis['pct_error'] = (df_test_analysis['abs_error'] / df_test_analysis['actual']) * 100

# Error by hour
error_by_hour = df_test_analysis.groupby('hour').agg({
    'abs_error': 'mean',
    'pct_error': 'mean'
}).reset_index()

# Error by day of week
error_by_dow = df_test_analysis.groupby('day_of_week').agg({
    'abs_error': 'mean',
    'pct_error': 'mean'
}).reset_index()

# Error by weekend
error_by_weekend = df_test_analysis.groupby('is_weekend').agg({
    'abs_error': 'mean',
    'pct_error': 'mean'
}).reset_index()
error_by_weekend['segment'] = error_by_weekend['is_weekend'].map({0: 'Weekday', 1: 'Weekend'})

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Error Analysis by Segments', fontsize=16, fontweight='bold')

# 1. Error by hour
axes[0, 0].bar(error_by_hour['hour'], error_by_hour['abs_error'], color='coral', edgecolor='black')
axes[0, 0].set_xlabel('Hour of Day')
axes[0, 0].set_ylabel('Mean Absolute Error (Bikes)')
axes[0, 0].set_title('Error by Hour of Day', fontweight='bold')
axes[0, 0].grid(True, alpha=0.3, axis='y')

# 2. Percentage error by hour
axes[0, 1].bar(error_by_hour['hour'], error_by_hour['pct_error'], color='lightblue', edgecolor='black')
axes[0, 1].set_xlabel('Hour of Day')
axes[0, 1].set_ylabel('Mean Percentage Error (%)')
axes[0, 1].set_title('Percentage Error by Hour', fontweight='bold')
axes[0, 1].grid(True, alpha=0.3, axis='y')

# 3. Error by day of week
day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
axes[1, 0].bar(day_labels, error_by_dow['abs_error'], color='green', edgecolor='black')
axes[1, 0].set_xlabel('Day of Week')
axes[1, 0].set_ylabel('Mean Absolute Error (Bikes)')
axes[1, 0].set_title('Error by Day of Week', fontweight='bold')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# 4. Weekday vs Weekend
axes[1, 1].bar(error_by_weekend['segment'], error_by_weekend['abs_error'], 
               color=['coral', 'lightblue'], edgecolor='black')
axes[1, 1].set_ylabel('Mean Absolute Error (Bikes)')
axes[1, 1].set_title('Weekday vs Weekend Error', fontweight='bold')
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"\n Error Analysis Insights:")
print(f"   Best prediction hours: {error_by_hour.nsmallest(3, 'abs_error')['hour'].tolist()}")
print(f"   Worst prediction hours: {error_by_hour.nlargest(3, 'abs_error')['hour'].tolist()}")
print(f"   Weekday MAE: {error_by_weekend[error_by_weekend['segment']=='Weekday']['abs_error'].values[0]:.2f}")
print(f"   Weekend MAE: {error_by_weekend[error_by_weekend['segment']=='Weekend']['abs_error'].values[0]:.2f}")


In [ ]:
def plot_error_heatmap(df_test):
    """
    Plots a heatmap of errors by Hour and Day of Week.
    Helps identify if the model fails specifically on 'Monday Mornings' etc.
    """
    # Pivot table for heatmap
    pivot_error = df_test.pivot_table(
        index='day_of_week', 
        columns='hour', 
        values='abs_error', 
        aggfunc='mean'
    )
    
    plt.figure(figsize=(12, 6))
    sns.heatmap(pivot_error, cmap='RdYlGn_r', annot=False)
    plt.title('Mean Absolute Error Heatmap (Day vs Hour)')
    plt.yticks(ticks=np.arange(7)+0.5, labels=['Mon','Tue','Wed','Thu','Fri','Sat','Sun'], rotation=0)
    plt.xlabel('Hour of Day')
    plt.show()

# Call this in your evaluation section
plot_error_heatmap(df_test_analysis)


##  SECTION 19: EXECUTIVE SUMMARY & KEY FINDINGS

Publication-ready summary of the entire analysis.

In [ ]:
print("\n" + "="*100)
print("EXECUTIVE SUMMARY - CAPITAL BIKESHARE DEMAND PREDICTION")
print("="*100)

print("\n DATASET CHARACTERISTICS:")
print(f"    Total Hours Analyzed:      {len(df_features):,}")
print(f"    Date Range:                {df_features['datetime_hourly'].min()} → {df_features['datetime_hourly'].max()}")
print(f"    COVID Years Removed:       2020-2021 (concept drift mitigation)")
print(f"    Training Samples:          {len(X_train):,} ({(1-CONFIG['test_size'])*100:.0f}%)")
print(f"    Test Samples:              {len(X_test):,} ({CONFIG['test_size']*100:.0f}%)")
print(f"    Features Engineered:       {len(feature_columns)}")

print(f"\n BEST MODEL PERFORMANCE (CatBoost):")
print(f"    R² Score:                  {results_cat['R2_real']:.4f} ({results_cat['R2_real']*100:.2f}% variance explained)")
print(f"    Adjusted R²:               {results_cat['Adj_R2']:.4f}")
print(f"    RMSE:                      {results_cat['RMSE_real']:.2f} bikes per hour")
print(f"    MAE:                       {results_cat['MAE_real']:.2f} bikes per hour")
print(f"    MAPE:                      {results_cat['MAPE']:.2f}%")

print(f"\n KEY SCIENTIFIC CONTRIBUTIONS:")
print(f"    1. COVID-Era Data Removal:")
# NOTE: rows_before/after were overwritten by the NaN cleanup step in the notebook. 
# Ideally, you should store the COVID counts in unique variables (e.g., rows_covid_before) to reference them here correctly.
print(f"       - Eliminated {rows_before - rows_after:,} rows (Note: reflects most recent drop operation)") 
print(f"       - Prevents model from learning pandemic-specific behaviors")
print(f"       - Improves generalization to normal operations")

print(f"\n    2. Data Leakage Prevention:")
print(f"       - Simulates real-world forecasting (using past weather)")
print(f"       - Prevents unrealistically high performance from same-day weather")

print(f"\n    3. Advanced Feature Engineering:")
print(f"       - Cyclical encoding (sin/cos) for temporal features")
print(f"       - Lag features (1h, 24h, 168h) for time series memory")
print(f"       - Rolling statistics (3h, 12h, 24h) for trend capture")
print(f"       - Domain-driven features (rush hour identification)")
print(f"       - Interaction terms (weather × time effects)")

print(f"\n    4. Time Series Cross-Validation:")
print(f"       - TimeSeriesSplit used (preserves chronological order)")
print(f"       - Prevents data leakage from future to past")
print(f"       - More rigorous than standard K-Fold CV")

print(f"\n    5. Comprehensive Ablation Study:")
# FIX APPLIED HERE: Used df_ablation instead of ablation_df and 'Impact (%)' instead of 'Impact_%'
ablation_sorted = df_ablation.iloc[1:].sort_values('Impact (%)', ascending=False)
for idx, row in ablation_sorted.iterrows():
    print(f"       - {row['Feature Group']}: +{row['Impact (%)']:.2f}% RMSE increase when removed")

print(f"\n FEATURE IMPORTANCE HIERARCHY:")
top_5_features = feature_importance.head(5)
for idx, row in top_5_features.iterrows():
    print(f"       {idx+1}. {row['feature']}: {row['importance']:.0f}")

print(f"\n MODEL COMPARISON:")
for idx, row in comparison_df.iterrows():
    print(f"    {row['Model']:20s}: R²={row['R²']:.4f}, RMSE={row['RMSE']:.2f}, MAPE={row['MAPE (%)']:.2f}%")

print(f"\n PRACTICAL INSIGHTS:")
print(f"    1. Past demand (lag features) is the strongest predictor")
print(f"    2. Rush hour patterns (7-9 AM, 4-7 PM) show distinct demand")
print(f"    3. Weather impacts demand, but less than temporal patterns")
print(f"    4. Model performs best during regular commuting hours")
print(f"    5. Weekday predictions are more accurate than weekends")

print("="*100)
print(" ANALYSIS COMPLETE - Model ready for deployment")
print("="*100)

##  SECTION 20: SAVE RESULTS 

Save model, predictions, and key metrics for future use.


In [ ]:

print("\n" + "="*70)
print("SAVING RESULTS")
print("="*70)

# Save comparison dataframe
comparison_df.to_csv('model_comparison.csv', index=False)
print(" Saved: model_comparison.csv")

# Save predictions
predictions_df = pd.DataFrame({
    'datetime': df_features.iloc[split_point:]['datetime_hourly'].values,
    'actual': y_test_real.values,
    'predicted': y_pred_best_real,
    'error': residuals,
    'abs_error': np.abs(residuals)
})
predictions_df.to_csv('predictions.csv', index=False)
print(" Saved: predictions.csv")

# Save ablation results
ablation_df.to_csv('ablation_study.csv', index=False)
print(" Saved: ablation_study.csv")

# Save feature importance
feature_importance.to_csv('feature_importance.csv', index=False)
print(" Saved: feature_importance.csv")

print("\n All results saved successfully!")
print("="*70)

In [ ]:
import joblib
import os
import shutil
from datetime import datetime

# 1. Create a directory to store models
save_dir = 'saved_models'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

print(f" Saving models to '{save_dir}/' directory...\n")

# 2. Dictionary of your potential models
# Format: {'Variable_Name': (Model_Object, 'Filename_Ext')}
# We use globals().get() to safely check if you have trained these models
models_to_save = {
    'Linear_Regression': (globals().get('lr_model'), 'pkl'),
    'Random_Forest': (globals().get('rf_model'), 'pkl'),
    'XGBoost': (globals().get('xgb_model'), 'json'),   # JSON is portable for XGB
    'LightGBM': (globals().get('lgb_model'), 'txt'),   # TXT is standard for LGBM
    'CatBoost': (globals().get('cat_model'), 'cbm'),   # CBM is native for CatBoost
    'Best_Tuned_Model': (globals().get('best_model'), 'cbm') # Assuming best_model is CatBoost
}

saved_count = 0

for name, (model, ext) in models_to_save.items():
    if model is not None:
        filepath = os.path.join(save_dir, f"{name}.{ext}")
        
        try:
            # Handle different save methods
            if name == 'CatBoost' or name == 'Best_Tuned_Model':
                # CatBoost native save
                model.save_model(filepath)
            elif name == 'XGBoost':
                # XGBoost native save (sklearn wrapper)
                model.save_model(filepath)
            elif name == 'LightGBM':
                # LightGBM native save (sklearn wrapper)
                # Note: booster_ returns the underlying engine
                model.booster_.save_model(filepath)
            else:
                # Scikit-Learn (LR, RF) use joblib
                joblib.dump(model, filepath)
                
            print(f" Saved {name} -> {filepath}")
            saved_count += 1
        except Exception as e:
            print(f" Could not save {name}: {str(e)}")
            # Fallback to joblib if native save fails
            try:
                joblib.dump(model, filepath.replace(ext, 'pkl'))
                print(f"   ↳ Fallback: Saved with joblib -> {filepath.replace(ext, 'pkl')}")
                saved_count += 1
            except:
                print("   ↳ Fallback failed.")

# 3. Zip the directory for easy download
if saved_count > 0:
    print("\n Zipping models...")
    shutil.make_archive('bikeshare_models', 'zip', save_dir)
    print(f" Success! Download 'bikeshare_models.zip' from the output directory.")
else:
    print("\n No models were found in memory to save.")